# Compare Fine-tuned Embedding Models — Zalo Legal Retrieval

Load `metrics_final_*.json` từ 4 training notebooks (01-04) → bảng so sánh + plots cho báo cáo.

**Setup**:
- Tạo notebook trên Kaggle, KHÔNG cần GPU (CPU đủ).
- Add Data: 4 Kaggle outputs từ notebook 01-04 (mỗi notebook output có file `metrics_final_<tag>.json`).
- Hoặc local: copy 4 JSON về `/kaggle/working/inputs/` rồi run.

## Cell 1 — Install deps

In [ ]:
!pip install -q pandas matplotlib seaborn tabulate

## Cell 2 — Discover + load JSON metrics

In [ ]:
import json
from pathlib import Path

# Auto-detect: tìm metrics_final_*.json trong /kaggle/input/ hoặc local
SEARCH_ROOTS = [Path("/kaggle/input"), Path("./inputs"), Path("/kaggle/working")]
metric_files = []
for root in SEARCH_ROOTS:
    if root.exists():
        metric_files.extend(root.rglob("metrics_final_*.json"))
metric_files = sorted(set(metric_files))
print(f"Found {len(metric_files)} metric files:")
for f in metric_files:
    print(f"  {f}")

assert metric_files, "Không tìm thấy metrics_final_*.json. Attach output từ 4 train notebooks vào Inputs."

results = {}
for path in metric_files:
    data = json.loads(path.read_text())
    tag = data["tag"]
    results[tag] = data
print(f"\nLoaded models: {list(results.keys())}")

## Cell 3 — Build comparison DataFrame

In [ ]:
import pandas as pd

# Bảng: rows = model × strategy, cols = các metric
rows = []
for tag, data in results.items():
    model_name = data["model_name"]
    # baseline
    row = {"model": tag, "strategy": "baseline", **{k: v for k, v in data["baseline"].items() if not k.startswith("_")}}
    rows.append(row)
    # inbatch best
    ib = data["inbatch"].get("best_metrics", {})
    if ib:
        row = {"model": tag, "strategy": f"inbatch (ep{data['inbatch'].get('best_epoch')})", **{k: v for k, v in ib.items() if not k.startswith("_") and k != "epoch"}}
        rows.append(row)
    # hardneg best
    hn = data["hardneg"].get("best_metrics", {})
    if hn:
        row = {"model": tag, "strategy": f"hardneg (ep{data['hardneg'].get('best_epoch')})", **{k: v for k, v in hn.items() if not k.startswith("_") and k != "epoch"}}
        rows.append(row)

df = pd.DataFrame(rows)
# Reorder columns: model, strategy, primary metrics first
primary = ["f2@10", "f2@5", "f2@1", "recall@10", "recall@5", "recall@1", "mrr", "ndcg@10", "precision@10", "precision@5", "precision@1"]
cols = ["model", "strategy"] + [c for c in primary if c in df.columns]
df = df[cols].round(4)
df

## Cell 4 — Markdown table for báo cáo

In [ ]:
from tabulate import tabulate
print(tabulate(df, headers="keys", tablefmt="github", showindex=False))

## Cell 5 — Bar chart: F2@10 by model × strategy

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")

# Pivot: model × strategy_type
df_plot = df.copy()
df_plot["strategy_short"] = df_plot["strategy"].apply(
    lambda s: "baseline" if s == "baseline" else "inbatch" if "inbatch" in s else "hardneg"
)
pivot = df_plot.pivot(index="model", columns="strategy_short", values="f2@10")
# Đặt cột theo thứ tự
cols = [c for c in ["baseline", "inbatch", "hardneg"] if c in pivot.columns]
pivot = pivot[cols]

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="bar", ax=ax, rot=15)
ax.set_ylabel("F2@10")
ax.set_title("F2@10 by model × training strategy (Zalo competition metric)")
ax.legend(title="Strategy")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.savefig("/kaggle/working/f2_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## Cell 6 — Convergence curves (F2@10 per epoch)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
colors = sns.color_palette("tab10", n_colors=len(results))

for color, (tag, data) in zip(colors, results.items()):
    # In-batch curve
    ib_eps = data["inbatch"].get("per_epoch", [])
    if ib_eps:
        epochs = [m["epoch"] for m in ib_eps]
        f2 = [m["f2@10"] for m in ib_eps]
        axes[0].plot(epochs, f2, marker="o", label=tag, color=color)
        # Mark best epoch
        be = data["inbatch"].get("best_epoch")
        if be:
            best_f2 = ib_eps[be - 1]["f2@10"]
            axes[0].scatter([be], [best_f2], s=150, edgecolors="black", linewidths=1.5, color=color, zorder=10)

    # Hard-neg curve
    hn_eps = data["hardneg"].get("per_epoch", [])
    if hn_eps:
        epochs = [m["epoch"] for m in hn_eps]
        f2 = [m["f2@10"] for m in hn_eps]
        axes[1].plot(epochs, f2, marker="s", label=tag, color=color)
        be = data["hardneg"].get("best_epoch")
        if be:
            best_f2 = hn_eps[be - 1]["f2@10"]
            axes[1].scatter([be], [best_f2], s=150, edgecolors="black", linewidths=1.5, color=color, zorder=10)

axes[0].set_title("In-batch negatives — F2@10 per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("F2@10")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title("Hard negatives (BM25, warm start) — F2@10 per epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/convergence_all.png", dpi=120, bbox_inches="tight")
plt.show()

## Cell 7 — Size vs Quality tradeoff

In [ ]:
# Model size mapping (MB)
MODEL_SIZES = {
    "bge-m3": 2270,
    "e5-multilingual-base": 470,
    "gte-multilingual-base": 660,
    "vietnamese-sbert": 120,
}

# Lấy winner F2@10 cho mỗi model
winner_data = []
for tag, data in results.items():
    winner_data.append({
        "model": tag,
        "size_mb": MODEL_SIZES.get(tag, 0),
        "winner_strategy": data.get("winner_strategy", "?"),
        "winner_f2@10": data.get("winner_metrics", {}).get("f2@10", 0.0),
    })
df_winner = pd.DataFrame(winner_data).sort_values("size_mb")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_winner["size_mb"], df_winner["winner_f2@10"], s=200)
for _, row in df_winner.iterrows():
    ax.annotate(f"{row['model']}\n({row['winner_strategy']})",
                (row["size_mb"], row["winner_f2@10"]),
                textcoords="offset points", xytext=(10, 5), fontsize=9)
ax.set_xlabel("Model size (MB)")
ax.set_ylabel("Winner F2@10")
ax.set_title("Model size vs winner F2@10 — tradeoff")
ax.grid(alpha=0.3)
ax.set_xscale("log")
plt.tight_layout()
plt.savefig("/kaggle/working/size_vs_quality.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nWinner table:")
print(tabulate(df_winner.round(4), headers="keys", tablefmt="github", showindex=False))

## Cell 8 — Overall winner pick

In [ ]:
best_tag = max(results.keys(), key=lambda t: results[t]["winner_metrics"].get("f2@10", 0.0))
best = results[best_tag]
print(f"{'='*60}")
print(f"OVERALL WINNER: {best_tag}")
print(f"Model:          {best['model_name']}")
print(f"Strategy:       {best['winner_strategy']} (epoch={best.get('winner_epoch')})")
print(f"F2@10:          {best['winner_metrics']['f2@10']:.4f}")
print(f"Recall@10:      {best['winner_metrics']['recall@10']:.4f}")
print(f"MRR:            {best['winner_metrics']['mrr']:.4f}")
print(f"nDCG@10:        {best['winner_metrics']['ndcg@10']:.4f}")
print(f"{'='*60}")
print(f"\nTo apply: set EMBEDDING_MODEL in backend/.env to your HF Hub repo (e.g., username/zalo-legal-{best_tag}-finetuned)")
print("Then re-run notebooks/kaggle_embed.ipynb to re-encode corpus into ChromaDB.")

## Cell 9 — Export combined results JSON

In [ ]:
combined = {
    "models": {tag: results[tag] for tag in results},
    "winner": {
        "tag": best_tag,
        "model_name": best["model_name"],
        "strategy": best["winner_strategy"],
        "metrics": best["winner_metrics"],
    },
    "comparison_table_rows": df.to_dict(orient="records"),
}
out_path = Path("/kaggle/working/combined_comparison.json")
out_path.write_text(json.dumps(combined, indent=2, ensure_ascii=False))
print(f"Saved: {out_path}")

# Also save markdown report
md_report = []
md_report.append("# Zalo Legal Retrieval — Model Comparison Report\n")
md_report.append(f"**Overall winner**: `{best_tag}` ({best['model_name']}) — strategy `{best['winner_strategy']}`\n")
md_report.append(f"- F2@10: **{best['winner_metrics']['f2@10']:.4f}**")
md_report.append(f"- Recall@10: {best['winner_metrics']['recall@10']:.4f}")
md_report.append(f"- MRR: {best['winner_metrics']['mrr']:.4f}\n")
md_report.append("## Comparison Table\n")
md_report.append(tabulate(df, headers="keys", tablefmt="github", showindex=False))
md_report.append("\n\n## Winner Apply Steps\n")
md_report.append(f"1. Push winner checkpoint to HF Hub from notebook {best_tag}.")
md_report.append(f"2. Update `backend/.env` → `EMBEDDING_MODEL=username/zalo-legal-{best_tag}-finetuned`.")
md_report.append("3. Re-run `notebooks/kaggle_embed.ipynb` để re-encode corpus.")
md_report.append("4. Re-deploy ChromaDB lên VPS.")

report_path = Path("/kaggle/working/report.md")
report_path.write_text("\n".join(md_report))
print(f"Saved: {report_path}")